
| | |
|---|---|
| **Nhóm** | 4 |
| **Thành viên** | 23730041 - Dương Thanh Quí |
| | 24730010 - Phan Tuấn Anh |
| | 24730004 - Hoàng Quốc Anh |
| **Lớp** | IE224.F21.CN1.CNTT |
| **Học phần** | Phân tích dữ liệu |
| **GVHD** | ThS. Phạm Thế Sơn |

# Đề tài: PHÂN TÍCH VÀ DỰ ĐOÁN GIÁ NHÀ SỬ DỤNG MÔ HÌNH HỒI QUY

## 1. Giới thiệu

Đề tài tập trung phân tích các yếu tố ảnh hưởng đến giá nhà và xây dựng
mô hình dự đoán giá dựa trên các đặc trưng như diện tích, số phòng ngủ,
số phòng tắm, tiện nghi và vị trí.

Bộ dữ liệu được sử dụng được thu thập từ
[Kaggle](https://www.kaggle.com/datasets/yasserh/housing-prices-dataset).

Mục tiêu: Xây dựng mô hình hồi quy dự đoán giá nhà (`price`) dựa trên
các thuộc tính còn lại.


In [ ]:
import pandas as pd

df = pd.read_csv("./data/Housing.csv")

## 2. Mô tả bộ dữ liệu

In [ ]:
df.shape

Bộ dữ liệu gồm 545 dòng, 13 cột.

In [ ]:
df.dtypes

Bộ dữ liệu gồm 13 cột:
- 7 cột số (int64): price, area, bedrooms, bathrooms, stories, parking
- 6 cột chữ (str): mainroad, guestroom, basement, hotwaterheating,
airconditioning, prefarea, furnishingstatus sẽ được encode ở bước tiền xử lý.

In [ ]:
df.isnull().sum()

Không có missing values, đây là một bộ dữ liệu sạch.

In [ ]:
df.describe().style.format('{:.2f}')

- Dataset có 545 dòng (count = 545)
- Giá nhà (price) trung bình ~4.77 triệu, dao động từ 1.75 triệu đến 13.3 triệu
- Diện tích (area) trung bình 5150, dao động từ 1650 đến 16200
- Số phòng ngủ (bedrooms) phổ biến là 3 (median = 3), tối đa 6
- Số phòng tắm (bathrooms) phổ biến là 1, tối đa 4
- Số tầng (stories) trung bình ~1.8, tối đa 4
- Bãi đậu xe (parking) phần lớn là 0 (median = 0), tối đa 3

In [ ]:
df.describe(include='str')

Nhận xét các cột chữ:
- 6 cột có 2 giá trị là chữ là: mainroad, guestroom, basement, hotwaterheating, airconditioning, prefarea
- 1 cột có 3 giá trị là chữ là: furnishingstatus

In [ ]:
for col in df.select_dtypes(include='str').columns:
    print(col, ':', df[col].unique())

Các giá trị cụ thể từng cột chữ:
- mainroad, guestroom, basement, hotwaterheating, airconditioning, prefarea: yes/no
- furnishingstatus: furnished, semi-furnished, unfurnished

## 3. Phương pháp phân tích

Trong phần này, nhóm trình bày các phương pháp được sử dụng để phân tích bộ dữ liệu và xây dựng mô hình dự đoán giá nhà.

Trước hết, dữ liệu được kiểm tra thêm về số lượng bản ghi trùng lặp, kiểu dữ liệu và đặc điểm của từng thuộc tính để xác định cách xử lý phù hợp. Sau đó, nhóm sử dụng thống kê mô tả kết hợp với trực quan hóa để hiểu rõ hơn về phân phối dữ liệu, phát hiện outlier và quan sát xu hướng của biến mục tiêu.

Đối với các biến số, nhóm sử dụng hệ số tương quan Pearson để đánh giá mức độ liên hệ với giá nhà. Đối với các biến phân loại, nhóm so sánh giá nhà giữa các nhóm thông qua bảng thống kê và biểu đồ boxplot.

Sau bước phân tích thăm dò, dữ liệu được mã hóa để phục vụ cho mô hình học máy. Cụ thể, các biến yes/no được chuyển về dạng 0/1 và biến `furnishingstatus` được one-hot encoding. Cuối cùng, nhóm sử dụng mô hình hồi quy tuyến tính đa biến để dự đoán giá nhà và đánh giá mô hình bằng các chỉ số `R²`, `RMSE` và `MAE`.


In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)


In [ ]:
duplicate_count = df.duplicated().sum()
print(f"Số dòng bị trùng lặp: {duplicate_count}")


## 4. Phân tích thăm dò dữ liệu

Phần này tập trung quan sát phân phối của các biến quan trọng, mức độ tương quan giữa các biến số và sự khác biệt về giá nhà theo từng nhóm thuộc tính.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df["price"], kde=True, ax=axes[0], color="#1f77b4")
axes[0].set_title("Phân phối giá nhà")
axes[0].set_xlabel("price")

sns.boxplot(x=df["price"], ax=axes[1], color="#9ecae1")
axes[1].set_title("Boxplot của giá nhà")
axes[1].set_xlabel("price")

plt.tight_layout()
plt.show()


Nhìn vào biểu đồ trên, có thể thấy biến `price` có độ phân tán khá lớn và xuất hiện một số giá trị cao vượt trội so với phần còn lại. Điều này cho thấy giá nhà trong bộ dữ liệu không phân bố hoàn toàn đồng đều và có dấu hiệu tồn tại outlier.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df["area"], kde=True, ax=axes[0], color="#2ca02c")
axes[0].set_title("Phân phối diện tích nhà")
axes[0].set_xlabel("area")

sns.boxplot(x=df["area"], ax=axes[1], color="#a1d99b")
axes[1].set_title("Boxplot của diện tích")
axes[1].set_xlabel("area")

plt.tight_layout()
plt.show()


Biến `area` có xu hướng lệch phải, nghĩa là phần lớn căn nhà có diện tích vừa phải, trong khi chỉ có một số ít căn nhà có diện tích rất lớn. Đặc điểm này thường làm cho giá nhà tăng mạnh ở nhóm có diện tích cao.


In [ ]:
numeric_cols = ["price", "area", "bedrooms", "bathrooms", "stories", "parking"]
corr_matrix = df[numeric_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap="Blues", fmt=".2f")
plt.title("Ma trận tương quan giữa các biến số")
plt.show()

corr_matrix["price"].sort_values(ascending=False)


Ma trận tương quan giúp nhóm xác định nhanh các biến số có liên hệ mạnh với `price`. Thông thường, `area`, `bathrooms`, `stories` và `parking` là các biến đáng chú ý vì có tương quan dương với giá nhà.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.scatterplot(data=df, x="area", y="price", ax=axes[0], color="#d62728")
axes[0].set_title("Mối quan hệ giữa diện tích và giá nhà")

sns.scatterplot(data=df, x="bathrooms", y="price", ax=axes[1], color="#9467bd")
axes[1].set_title("Mối quan hệ giữa số phòng tắm và giá nhà")

plt.tight_layout()
plt.show()


Từ scatter plot có thể thấy giá nhà có xu hướng tăng khi diện tích tăng. Bên cạnh đó, số phòng tắm nhiều hơn cũng thường đi kèm với mức giá cao hơn, dù dữ liệu vẫn có độ phân tán tương đối lớn.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=df, x="airconditioning", y="price", ax=axes[0], palette="Set2")
axes[0].set_title("Giá nhà theo tình trạng điều hòa")
axes[0].set_xlabel("airconditioning")
axes[0].set_ylabel("price")

sns.boxplot(data=df, x="furnishingstatus", y="price", ax=axes[1], palette="Set3")
axes[1].set_title("Giá nhà theo tình trạng nội thất")
axes[1].set_xlabel("furnishingstatus")
axes[1].set_ylabel("price")

plt.tight_layout()
plt.show()


Các căn nhà có `airconditioning = yes` thường có mức giá cao hơn nhóm không có điều hòa. Ngoài ra, nhóm `furnished` cũng có xu hướng có giá cao hơn so với `semi-furnished` và `unfurnished`, cho thấy tiện nghi và mức độ hoàn thiện nội thất có ảnh hưởng nhất định đến giá bán.


In [ ]:
outlier_summary = []
for col in numeric_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    outlier_count = ((df[col] < lower_bound) | (df[col] > upper_bound)).sum()
    outlier_summary.append({
        "column": col,
        "Q1": q1,
        "Q3": q3,
        "IQR": iqr,
        "lower_bound": lower_bound,
        "upper_bound": upper_bound,
        "outlier_count": outlier_count,
    })

outlier_df = pd.DataFrame(outlier_summary)
outlier_df


Bảng trên cho thấy một số biến như `price`, `area` hoặc `stories` có xuất hiện outlier theo quy tắc IQR. Tuy nhiên, đây vẫn có thể là các giá trị hợp lệ trong thực tế nên nhóm giữ lại dữ liệu và tiếp tục đánh giá tác động của chúng trong mô hình hồi quy.


Từ kết quả phân tích sơ bộ, có thể thấy một số thuộc tính như `area`, `bathrooms`, `airconditioning` và `prefarea` có khả năng ảnh hưởng đến giá nhà. Vì vậy, nhóm lựa chọn mô hình hồi quy tuyến tính đa biến để xây dựng mô hình dự đoán.


## 5. Xây dựng mô hình hồi quy

Ở bước này, nhóm tiến hành mã hóa các biến phân loại, chia dữ liệu thành tập huấn luyện và tập kiểm tra, sau đó huấn luyện mô hình hồi quy tuyến tính đa biến.


In [ ]:
df_model = df.copy()

binary_cols = [
    "mainroad",
    "guestroom",
    "basement",
    "hotwaterheating",
    "airconditioning",
    "prefarea",
]

for col in binary_cols:
    df_model[col] = df_model[col].map({"yes": 1, "no": 0})

df_model = pd.get_dummies(df_model, columns=["furnishingstatus"], drop_first=True, dtype=int)

df_model.head()


In [ ]:
X = df_model.drop(columns=["price"])
y = df_model["price"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

linear_model = LinearRegression()
linear_model.fit(X_train, y_train)

y_pred = linear_model.predict(X_test)


In [ ]:
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

metric_df = pd.DataFrame(
    {
        "Chỉ số": ["MAE", "RMSE", "R²"],
        "Giá trị": [mae, rmse, r2],
    }
)
metric_df


Các chỉ số `MAE`, `RMSE` và `R²` được sử dụng để đánh giá chất lượng mô hình. Nếu `R²` ở mức khá và sai số không quá lớn, có thể xem mô hình hồi quy tuyến tính là một lựa chọn phù hợp để dự đoán giá nhà trong bộ dữ liệu này.


In [ ]:
coef_df = pd.DataFrame(
    {
        "Biến": X.columns,
        "Hệ số hồi quy": linear_model.coef_,
    }
).sort_values(by="Hệ số hồi quy", key=lambda s: s.abs(), ascending=False)

coef_df


Bảng hệ số hồi quy giúp nhóm quan sát mức độ tác động của từng thuộc tính lên giá nhà. Các biến có trị tuyệt đối hệ số lớn thường là các yếu tố ảnh hưởng mạnh hơn đến kết quả dự đoán.


In [ ]:
plt.figure(figsize=(7, 7))
sns.scatterplot(x=y_test, y=y_pred, color="#1f77b4")
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], color="red", linestyle="--")
plt.xlabel("Giá thực tế")
plt.ylabel("Giá dự đoán")
plt.title("So sánh giá thực tế và giá dự đoán")
plt.show()


In [ ]:
residuals = y_test - y_pred

plt.figure(figsize=(10, 5))
sns.histplot(residuals, kde=True, color="#ff7f0e")
plt.title("Phân phối phần dư của mô hình")
plt.xlabel("Residuals")
plt.show()


## 6. Kết quả phân tích

Qua quá trình phân tích, nhóm nhận thấy diện tích, số phòng tắm, vị trí ưu tiên và các tiện nghi như điều hòa có mối liên hệ tích cực với giá nhà. Mô hình hồi quy tuyến tính đa biến cho phép lượng hóa các mối quan hệ này và cung cấp khả năng dự đoán giá nhà ở mức chấp nhận được.

Bên cạnh đó, dữ liệu vẫn tồn tại outlier và độ phân tán khá lớn ở biến `price`, nên kết quả dự đoán có thể bị ảnh hưởng ở một số trường hợp đặc biệt. Tuy vậy, mô hình vẫn phù hợp để làm mô hình nền cho bài toán dự đoán giá nhà.


## 7. Kết luận

Bài toán phân tích và dự đoán giá nhà đã cho thấy dữ liệu bất động sản chịu ảnh hưởng bởi nhiều yếu tố khác nhau, trong đó diện tích và tiện nghi là các yếu tố nổi bật. Thông qua quá trình phân tích thăm dò và xây dựng mô hình hồi quy, nhóm có thể rút ra được các đặc trưng quan trọng và áp dụng chúng để dự đoán giá nhà.

Trong tương lai, nhóm có thể thử thêm các mô hình khác như Ridge, Lasso hoặc Random Forest để so sánh hiệu quả dự đoán và cải thiện độ chính xác của hệ thống.
